# Sampling debug

학습 첫 iteration에 C++가 덤프한 샘플링 결과를 불러와 시각화한다.

- **ray marching**: `LocalMap::sample` (occupied voxel 통과 점)
- **표면 근처 샘플링**: `utils::sample_surface_pts`
- **sphere tracing**: 이미지 렌더링 학습 시 `tracer::render_ray`

덤프 위치: `<output_run>/sample_debug/*.bin` (포맷: int64 ndim; int64 shape[]; float32 data)

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

def load_bin(path):
    with open(path, 'rb') as f:
        ndim = int(np.fromfile(f, '<i8', 1)[0])
        shape = np.fromfile(f, '<i8', ndim).astype(int)
        return np.fromfile(f, '<f4', int(np.prod(shape))).reshape(shape)

def load_dir(d):
    out = {}
    for p in glob.glob(os.path.join(d, '*.bin')):
        out[os.path.splitext(os.path.basename(p))[0]] = load_bin(p)
    return out

def sub(a, n=20000):
    if len(a) <= n:
        return a
    return a[np.random.choice(len(a), n, replace=False)]

In [ ]:
# === 덤프 폴더 지정 (학습 output run 의 sample_debug) ===
DEBUG_DIR = 'output/latest_run/sample_debug'   # <- 본인 경로로 수정
D = load_dir(DEBUG_DIR)
print('loaded from', DEBUG_DIR)
for k, v in sorted(D.items()):
    print(f'  {k:22s} {v.shape}')

In [ ]:
# === ray marching + 표면 근처 샘플링 ===
fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d')

if 'raymarch_xyz' in D:
    p = D['raymarch_xyz']
    idx = np.random.choice(len(p), min(len(p), 20000), replace=False)
    c = D['raymarch_raysdf'][idx, 0] if 'raymarch_raysdf' in D else 'tab:blue'
    sc = ax.scatter(p[idx, 0], p[idx, 1], p[idx, 2], c=c, s=2,
                    cmap='coolwarm', label='raymarch (color=ray_sdf)')
    if 'raymarch_raysdf' in D:
        plt.colorbar(sc, ax=ax, shrink=0.5, label='ray_sdf')

if 'surface_xyz' in D:
    p = sub(D['surface_xyz'])
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c='lime', s=3, label='surface samples')

if 'ray_xyz' in D:
    p = sub(D['ray_xyz'], 5000)
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c='k', s=6, label='ray hit (lidar surf)')

# ray 선분 (origin -> origin + dir*depth)
if all(k in D for k in ('ray_origin', 'ray_direction', 'ray_depth')):
    o, dd, t = D['ray_origin'], D['ray_direction'], D['ray_depth']
    t = t.reshape(len(t), -1)[:, :1]
    for i in np.random.choice(len(o), min(len(o), 60), replace=False):
        e = o[i] + dd[i] * t[i]
        ax.plot([o[i, 0], e[0]], [o[i, 1], e[1]], [o[i, 2], e[2]],
                c='gray', lw=0.5, alpha=0.5)

ax.legend(); ax.set_title('ray marching + surface sampling')
plt.tight_layout(); plt.show()

In [ ]:
# === sphere tracing (이미지 렌더링) ===
fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d')

if 'strace_pts' in D:
    p = D['strace_pts']
    idx = np.random.choice(len(p), min(len(p), 30000), replace=False)
    c = D['strace_depth'][idx, 0] if 'strace_depth' in D else 'r'
    sc = ax.scatter(p[idx, 0], p[idx, 1], p[idx, 2], c=c, s=2, cmap='viridis')
    if 'strace_depth' in D:
        plt.colorbar(sc, ax=ax, shrink=0.5, label='depth')

if all(k in D for k in ('strace_ray_origin', 'strace_ray_direction')):
    o, dd = D['strace_ray_origin'], D['strace_ray_direction']
    for i in np.random.choice(len(o), min(len(o), 60), replace=False):
        e = o[i] + dd[i] * 5.0
        ax.plot([o[i, 0], e[0]], [o[i, 1], e[1]], [o[i, 2], e[2]],
                c='gray', lw=0.4, alpha=0.4)

ax.set_title('sphere tracing samples (color=depth)')
plt.tight_layout(); plt.show()

In [ ]:
# === (선택) open3d/CloudCompare 용 ply 내보내기 ===
import sys
sys.path.insert(0, '../scripts/visual_feature')
from camera_utils import write_ply
import matplotlib.cm as cm

def turbo(x):
    x = (x - x.min()) / (np.ptp(x) + 1e-9)
    return (cm.turbo(x)[:, :3] * 255).astype('uint8')

od = os.path.join(DEBUG_DIR, 'ply'); os.makedirs(od, exist_ok=True)
if 'raymarch_xyz' in D:
    c = turbo(D['raymarch_raysdf'][:, 0]) if 'raymarch_raysdf' in D else None
    write_ply(os.path.join(od, 'raymarch.ply'), D['raymarch_xyz'], c)
if 'surface_xyz' in D:
    write_ply(os.path.join(od, 'surface.ply'), D['surface_xyz'])
if 'strace_pts' in D:
    c = turbo(D['strace_depth'][:, 0]) if 'strace_depth' in D else None
    write_ply(os.path.join(od, 'strace.ply'), D['strace_pts'], c)
print('ply written ->', od)